# 4. Run an experiment for the installed method

Experiments can run for hours, so the runner is started in the background on `control` (it keeps running if
this notebook or the Jupyter session is closed). Use the status cell to follow progress and the download cell
to copy the results into `results/` next to this notebook.

In [ ]:
METHOD = "baseline"   # must match the method installed by 03_install_method
CYCLES = None         # None = runner default (120 local steps; 12 rounds for fed-encrypt-dynamic)
RUNS = 1
EXTRA_ARGS = ""       # e.g. "--communication-frequency 10 --sample-batch-size 64" or "--policy-removal 4 --policy-reintroduction 8"

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

In [ ]:
import datetime
import shlex

assert METHOD in cfg.METHODS, f"Unknown method {METHOD!r}"
installed = run(control, f"cat {cfg.INSTALLED_METHOD_FILE} 2>/dev/null || true", quiet=True).strip()
assert installed == METHOD, f"Installed method is {installed or 'none'!r}, not {METHOD!r}; run 03_install_method first"

checkout = cfg.checkout_dir(METHOD)
python = f"$HOME/{cfg.EXPERIMENT_VENV}/bin/python"
run(control, f"{python} -m pip install -q -r $HOME/{checkout}/{cfg.METHODS[METHOD]['requirements']}", quiet=True)
print(run(control, f"{python} $HOME/{checkout}/{cfg.METHODS[METHOD]['runner']} --help", quiet=True))

## Start the experiment in the background

In [ ]:
running = run(control, f"test -f {cfg.CURRENT_RUN_FILE} && kill -0 $(cut -d' ' -f2 {cfg.CURRENT_RUN_FILE}) 2>/dev/null && echo yes || echo no", quiet=True).strip()
assert running == "no", "An experiment is still running; check or stop it below first"

run_id = f"{METHOD}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
output_dir = f"$HOME/{cfg.RESULTS_DIR}/{run_id}"
log_file = f"$HOME/{cfg.LOGS_DIR}/{run_id}.log"
args = f"--runs {RUNS} --output-dir {output_dir}"
if CYCLES:
    args += f" --cycles {CYCLES}"
if EXTRA_ARGS:
    args += f" {EXTRA_ARGS}"

start = (
    f"mkdir -p $HOME/{cfg.RESULTS_DIR} $HOME/{cfg.LOGS_DIR} && cd $HOME/{checkout} || exit 1; "
    f"nohup setsid {python} {cfg.METHODS[METHOD]['runner']} {args} > {log_file} 2>&1 < /dev/null & "
    f"echo \"{run_id} $!\" > $HOME/{cfg.CURRENT_RUN_FILE}; sleep 5; kill -0 $! || {{ tail -n 30 {log_file}; exit 1; }}"
)
run(control, start, quiet=True)
print(f"Started {run_id}: {cfg.METHODS[METHOD]['runner']} {args}")

## Status (rerun whenever you like)

In [ ]:
print(run(control, f"""
read run_id pid < {cfg.CURRENT_RUN_FILE}
if kill -0 "$pid" 2>/dev/null; then echo "$run_id: RUNNING (pid $pid)"; else echo "$run_id: FINISHED"; fi
echo; tail -n 30 $HOME/{cfg.LOGS_DIR}/$run_id.log
""", quiet=True))

## Stop the running experiment (only if needed)

In [ ]:
STOP = False

if STOP:
    print(run(control, f"read run_id pid < {cfg.CURRENT_RUN_FILE}; kill -TERM -- -$pid && echo stopped $run_id", check=False, quiet=True))

## Download results and log into `results/`

In [ ]:
import os
import tarfile

run_id = run(control, f"cut -d' ' -f1 {cfg.CURRENT_RUN_FILE}", quiet=True).strip()
archive = f"{cfg.REMOTE_TOOLS_DIR}/{run_id}.tar.gz"
run(control, f"cd $HOME && tar -czf {archive} --ignore-failed-read {cfg.RESULTS_DIR}/{run_id} {cfg.LOGS_DIR}/{run_id}.log", quiet=True)

os.makedirs("results", exist_ok=True)
local_archive = os.path.join("results", f"{run_id}.tar.gz")
control.download_file(local_file_path=local_archive, remote_file_path=archive)
with tarfile.open(local_archive) as tar:
    tar.extractall(os.path.join("results", run_id))
print(f"Saved to results/{run_id}/")